## 1. Import Libraries

In [14]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

## 2. Load Dataset

In [15]:
# Load dataset

df = pd.read_csv('dataset/air_quality/air_quality.csv')

# Show first rows

df.head()

## 3. Dataset Information

In [16]:
print(df.shape)
print(df.columns)
# Check missing values

df.isnull().sum()


## 4. Convert Time Column

In [17]:
# Convert datetime column

df['ts_utc'] = pd.to_datetime(df['ts_utc'])

# Sort by time

df = df.sort_values('ts_utc')

# Reset index

df.reset_index(drop=True, inplace=True)

## 5. Select Target Column

In [18]:
# Select AQI column

target_col = 'aqi'

series = df[target_col]

## 6. Visualize Time Series

In [19]:
plt.figure(figsize=(15,5))
plt.plot(df['ts_utc'], series)
plt.title('AQI Time Series')
plt.xlabel('Time')
plt.ylabel('AQI')
plt.show()

## 7. Train / Validation / Test Split (70/20/10)

In [20]:
n = len(df)

train_size = int(n * 0.7)
val_size   = int(n * 0.2)

test_size  = n - train_size - val_size

train = series[:train_size]
val   = series[train_size:train_size+val_size]
test  = series[train_size+val_size:]

print('Train size:', len(train))
print('Validation size:', len(val))
print('Test size:', len(test))

## 8. Plot Dataset Split

In [21]:
plt.figure(figsize=(15,5))

plt.plot(train.index, train, label='Train')
plt.plot(val.index, val, label='Validation')
plt.plot(test.index, test, label='Test')

plt.legend()
plt.title('Dataset Split')
plt.show()

## Silding window configuration

In [47]:
SEQ_LEN  = 96   # số điểm input (lookback)
PRED_LEN = 12   # số điểm cần dự báo (horizon)
STRIDE   = 12   # bước trượt

print(f'SEQ_LEN  : {SEQ_LEN}')
print(f'PRED_LEN : {PRED_LEN}')
print(f'STRIDE   : {STRIDE}')

## 9. Evaluation Metrics Function

In [48]:
def evaluate(y_true, y_pred, model_name='Model'):
    """
    y_true, y_pred: 1-D array — toàn bộ các điểm predict ghép lại từ tất cả windows.
    """
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)

    print(f'[{model_name}]')
    print(f'  MAE  : {mae:.6f}')
    print(f'  MSE  : {mse:.6f}')
    print(f'  RMSE : {rmse:.6f}')
    print(f'  R2   : {r2:.6f}')

    return mae, mse, rmse, r2

## silding window generator

In [49]:
def sliding_windows(series_array, seq_len, pred_len, stride):
    """
    Tạo danh sách các (input_window, target_window) từ một mảng 1-D.

    Parameters
    ----------
    series_array : np.ndarray  — toàn bộ chuỗi
    seq_len      : int         — độ dài input
    pred_len     : int         — độ dài cần dự báo
    stride       : int         — bước trượt

    Returns
    -------
    inputs  : list of np.ndarray, mỗi phần tử shape (seq_len,)
    targets : list of np.ndarray, mỗi phần tử shape (pred_len,)
    """
    inputs, targets = [], []
    total = len(series_array)

    start = 0
    while start + seq_len + pred_len <= total:
        x = series_array[start : start + seq_len]
        y = series_array[start + seq_len : start + seq_len + pred_len]
        inputs.append(x)
        targets.append(y)
        start += stride

    return inputs, targets


# Ghép train[-SEQ_LEN:] + toàn bộ test
# → window đầu tiên dùng đúng 96 điểm lịch sử từ train
context = np.concatenate([train.values[-SEQ_LEN:], test.values])

test_inputs, test_targets = sliding_windows(context, SEQ_LEN, PRED_LEN, STRIDE)

print(f'Số windows          : {len(test_inputs)}')
print(f'Input shape / window: {test_inputs[0].shape}')   # (96,)
print(f'Target shape/ window: {test_targets[0].shape}')  # (24,)

## 10. Naive Forecast Model

In [50]:
naive_preds   = []
naive_targets = []

for x, y in zip(test_inputs, test_targets):
    yhat = np.full(PRED_LEN, x[-1])   # giá trị cuối lặp 24 lần
    naive_preds.append(yhat)
    naive_targets.append(y)

naive_preds   = np.concatenate(naive_preds)
naive_targets = np.concatenate(naive_targets)

## Evaluate Naive Model

In [51]:
evaluate(naive_targets, naive_preds, 'Naive')

## Plot Naive Forecast

In [52]:
plt.figure(figsize=(12, 4))
plt.plot(range(PRED_LEN), test_targets[0], label='True')
plt.plot(range(PRED_LEN), naive_preds[:PRED_LEN], label='Naive Prediction')
plt.legend()
plt.title('Naive Forecast — Window 1')
plt.xlabel('Step ahead')
plt.ylabel('AQI')
plt.show()

## 11. Moving Average Model

In [53]:
WINDOW_SIZE = 5

ma_preds   = []
ma_targets = []

for x, y in zip(test_inputs, test_targets):
    yhat = np.full(PRED_LEN, np.mean(x[-WINDOW_SIZE:]))
    ma_preds.append(yhat)
    ma_targets.append(y)

ma_preds   = np.concatenate(ma_preds)
ma_targets = np.concatenate(ma_targets)

## Evaluate Moving Average

In [54]:
evaluate(ma_targets, ma_preds, 'Moving Average')

## Plot Moving Average Forecast

In [56]:
plt.figure(figsize=(12, 4))
plt.plot(range(PRED_LEN), test_targets[0], label='True')
plt.plot(range(PRED_LEN), ma_preds[:PRED_LEN], label='Moving Average Prediction')
plt.legend()
plt.title('Moving Average Forecast — Window 1')
plt.xlabel('Step ahead')
plt.ylabel('AQI')
plt.show()

## 12. Check Stationarity for ARIMA/SARIMA
Augmented Dickey-Fuller (ADF) Test

In [57]:
result = adfuller(train)
print('ADF Statistic:', result[0])
print('p-value      :', result[1])
print('=> Stationary' if result[1] < 0.05 else '=> Non-stationary (cần differencing)')

Interpretation
p-value < 0.05 → stationary
p-value > 0.05 → non-stationary

## 13. Differencing

In [32]:
train_diff = train.diff().dropna()

## Check Stationarity Again

In [33]:
result_diff = adfuller(train_diff)

print('ADF Statistic:', result_diff[0])
print('p-value:', result_diff[1])

## 14. ARIMA Model
ARIMA(p,d,q)

Example:

p = 2
d = 1
q = 2

In [ ]:
arima_preds   = []
arima_targets = []

for i, (x, y) in enumerate(zip(test_inputs, test_targets)):
    try:
        model = ARIMA(x, order=(2, 0, 2))
        fit   = model.fit()
        yhat  = fit.forecast(steps=PRED_LEN)
    except Exception:
        # Fallback: naive nếu ARIMA không hội tụ
        yhat = np.full(PRED_LEN, x[-1])

    arima_preds.append(yhat)
    arima_targets.append(y)

    if (i + 1) % 5 == 0 or i == 0:
        print(f'  Window {i+1}/{len(test_inputs)} done')

arima_preds   = np.concatenate(arima_preds)
arima_targets = np.concatenate(arima_targets)
print('ARIMA done.')

## Evaluate ARIMA

In [44]:
evaluate(arima_targets, arima_preds, 'ARIMA')

In [60]:
# In thử 3 window đầu để xem pattern
for i in range(3):
    pred_window = arima_preds[i*24:(i+1)*24]
    true_window = arima_targets[i*24:(i+1)*24]
    print(f'\nWindow {i+1}')
    print(f'  True  : min={true_window.min():.1f}, max={true_window.max():.1f}, mean={true_window.mean():.1f}')
    print(f'  Pred  : min={pred_window.min():.1f}, max={pred_window.max():.1f}, mean={pred_window.mean():.1f}')


Window 1
  True  : min=62.0, max=93.0, mean=70.1
  Pred  : min=57.8, max=99.2, mean=69.6

Window 2
  True  : min=63.0, max=89.0, mean=67.9
  Pred  : min=63.5, max=90.1, mean=68.4

Window 3
  True  : min=61.0, max=85.0, mean=70.1
  Pred  : min=60.9, max=77.8, mean=69.3


## Plot ARIMA Forecast

In [45]:
plt.figure(figsize=(12, 4))
plt.plot(range(PRED_LEN), test_targets[0], label='True')
plt.plot(range(PRED_LEN), arima_preds[:PRED_LEN], label='ARIMA Prediction')
plt.legend()
plt.title('ARIMA Forecast — Window 1')
plt.xlabel('Step ahead')
plt.ylabel('AQI')
plt.show()

## 15. SARIMA Model
SARIMA(p,d,q)(P,D,Q,s)

Example:

Seasonal period s = 24
Suitable for hourly data

In [37]:
sarima_preds   = []
sarima_targets = []

for i, (x, y) in enumerate(zip(test_inputs, test_targets)):
    try:
        model = SARIMAX(
            x,
            order=(2, 0, 2),
            seasonal_order=(1, 1, 1, 24)
        )
        fit  = model.fit(disp=False)
        yhat = fit.forecast(steps=PRED_LEN)
    except Exception:
        # Fallback: naive nếu SARIMA không hội tụ
        yhat = np.full(PRED_LEN, x[-1])

    sarima_preds.append(yhat)
    sarima_targets.append(y)

    if (i + 1) % 5 == 0 or i == 0:
        print(f'  Window {i+1}/{len(test_inputs)} done')

sarima_preds   = np.concatenate(sarima_preds)
sarima_targets = np.concatenate(sarima_targets)
print('SARIMA done.')

## Evaluate SARIMA

In [38]:
evaluate(sarima_targets, sarima_preds, 'SARIMA')

## Plot SARIMA Forecast

In [39]:
plt.figure(figsize=(12, 4))
plt.plot(range(PRED_LEN), test_targets[0], label='True')
plt.plot(range(PRED_LEN), sarima_preds[:PRED_LEN], label='SARIMA Prediction')
plt.legend()
plt.title('SARIMA Forecast — Window 1')
plt.xlabel('Step ahead')
plt.ylabel('AQI')
plt.show()

## 16. Compare All Models

In [40]:
results = []

mae, mse, rmse, r2 = evaluate(naive_targets,  naive_preds,  'Naive')
results.append(['Naive', mae, mse, rmse, r2])

mae, mse, rmse, r2 = evaluate(ma_targets,     ma_preds,     'Moving Average')
results.append(['Moving Average', mae, mse, rmse, r2])

mae, mse, rmse, r2 = evaluate(arima_targets,  arima_preds,  'ARIMA')
results.append(['ARIMA', mae, mse, rmse, r2])

mae, mse, rmse, r2 = evaluate(sarima_targets, sarima_preds, 'SARIMA')
results.append(['SARIMA', mae, mse, rmse, r2])

## Create Comparison Table

In [41]:
results_df = pd.DataFrame(
    results,
    columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2']
)

results_df

## 17. Visualization Comparison

In [42]:
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(results_df['Model'], results_df['RMSE'], color=colors)
axes[0].set_title('RMSE Comparison (lower is better)')
axes[0].set_ylabel('RMSE')

axes[1].bar(results_df['Model'], results_df['R2'], color=colors)
axes[1].set_title('R² Comparison (higher is better)')
axes[1].set_ylabel('R²')

plt.tight_layout()
plt.show()